**imports**
- sys.path is the list of folders where Python looks for modules
- The notebook runs from within the `notebooks/` directory, and `src/` is one level up (`..`), so that folder needs to be added.

In [5]:
import sys
sys.path.append("..")          # the notebook runs inside notebooks/; this lets Python find src/

import numpy as np
import pandas as pd
from src.cashflow import load_constants
from src.cashflow import run_trajectory
from src.bootstrap import block_bootstrap

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


**Comparison: real vs theoretical autocorrelation**
- Real autocorrelation: e_base.autocorr(lag=k)
- theoretical autocorrelation: 0.64 ** k

In [6]:
#1. Upload csv file
df = pd.read_csv(r"../data/processed/annual_energy.csv")
e_base = df ['energy_base_gwh']

#2. Create the list with rows in 'for' loop
rows = []
for k in range(1, 9):
    rows.append({
        "lag": k, 
        "real": e_base.autocorr(lag=k),
        "ar1": 0.64 ** k})

#3. Create and show dataframe
acf = pd.DataFrame(rows)
acf

,lag,real,ar1
0,1,0.644530,0.640000
1,2,0.466191,0.409600
2,3,0.352540,0.262144
3,4,0.321592,0.167772
4,5,0.253385,0.107374
5,6,0.240802,0.068719
6,7,0.213756,0.043980
7,8,0.208350,0.028147


**Interpretation**:
- The “each year depends only on the previous one” model underestimates the river’s memory. When accounting for actual autocorrelation, the drop below 0.236 does not occur until 6 to 7 years later, not at 3 or 4 years. The river possesses a slow, multi-year component that the 0.64^k term fails to capture.

**Choosing the new L value**:
We need to consider that:
- The 0.236 band is thinner than it looks.
- There is a third, automatic criterion, which is the standard one. The Politis and White method estimates the optimal block length directly from the series.

In [7]:
from arch.bootstrap import optimal_block_length
optimal_block_length(df['energy_base_gwh'])


,stationary,circular
energy_base_gwh,6.979152,7.989135


**Result summary to choose L**
| Method | Suggested L | Observation |
| :--- | :---: | :---: |
| n^(1/3) | 4   | This method does not have any constant that depends of the data serie |
| Actual autocorrelation vs. band | 6 - 7 | The band is not that precise |
| Politis-White (circular) | 8 | Published method, automatic, without estimated ranges |

In [8]:
semi = pd.read_csv("../data/processed/semiannual_energy.csv")
table = semi.pivot(index="hydro_year", columns="half", values="energy_base_gwh").to_numpy()

# Parameters
n_records = table.shape[0]                           # (69, 2): one row per hydrological year, columns S1 and S2
n_years = 30 
L=8 

# Random number generator
rng = np.random.default_rng(seed=42)

# Run a complete trajectory
    # Positions
positions = block_bootstrap(n_records, n_years, L, rng)
    # Extract the sorted rows
sorted_years_table = table[positions]
    # flatten row pwe row to obtain the cronological serie of 60 semesters
energy_trajectory = sorted_years_table.ravel()

# Execute funtion run_trajectory
c = load_constants("../model/hydro_pf_model_v1.xlsx")
result = run_trajectory(energy_trajectory, c )

print(f"Table extracted per year: {sorted_years_table.shape}")  # Should be (30, 2)
print(f"Length of the final energy vector: {len(energy_trajectory)}")  # Should be 60
print("\nFirst 6 values of the vector (S1 year  1, S2 year 1, S1 year 2...):")
print(energy_trajectory[:6])

print(f"Minimum DSCR (12m) = ",(result["dscr_12m"][c["repayment_flag"] == 1].min()))
print(f"Equity IRR = " , result["equity_irr"])
print(f"Quantity of lockups = ", result["lockup"].sum())
print(f"Quantity of defaults = ", result["default"].sum())

Table extracted per year: (30, 2)
Length of the final energy vector: 60

First 6 values of the vector (S1 year  1, S2 year 1, S1 year 2...):
[34.72624783 14.31234707 33.77413039 18.37214137 29.34512023 14.35140804]
Minimum DSCR (12m) =  0.775677637420131
Equity IRR =  0.03045651571905328
Quantity of lockups =  2.0
Quantity of defaults =  2.0
